In [1]:
import os
import requests
import zipfile
import pandas as pd
import xml.etree.ElementTree as ET

print("LankaLandslide AI")
print("Landslide data collection started.")

LankaLandslide AI
Landslide data collection started.


In [2]:
BASE_DIR = os.path.abspath("..")

RAW_LANDSLIDE_DIR = os.path.join(
    BASE_DIR, "data", "raw", "landslides"
)

PROCESSED_DIR = os.path.join(
    BASE_DIR, "data", "processed"
)

os.makedirs(RAW_LANDSLIDE_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Project directory:")
print(BASE_DIR)

print("\nRaw landslide directory:")
print(RAW_LANDSLIDE_DIR)

print("\nProcessed directory:")
print(PROCESSED_DIR)

Project directory:
c:\Users\User\Documents\GitHub\LankaLandslide-AI

Raw landslide directory:
c:\Users\User\Documents\GitHub\LankaLandslide-AI\data\raw\landslides

Processed directory:
c:\Users\User\Documents\GitHub\LankaLandslide-AI\data\processed


In [3]:
url = "https://www.desinventar.net/DesInventar/download/DI_export_lka.zip"

zip_path = os.path.join(
    RAW_LANDSLIDE_DIR,
    "DI_export_lka.zip"
)

print("Downloading Sri Lankan landslide dataset...")

response = requests.get(url, timeout=120)

response.raise_for_status()

with open(zip_path, "wb") as f:
    f.write(response.content)

print("Download completed.")
print("Saved to:")
print(zip_path)

Download completed.
Saved to:
c:\Users\User\Documents\GitHub\LankaLandslide-AI\data\raw\landslides\DI_export_lka.zip


In [4]:
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(RAW_LANDSLIDE_DIR)

print("Dataset extracted successfully.")

print("\nFiles:")
for file in os.listdir(RAW_LANDSLIDE_DIR):
    print(file)

Dataset extracted successfully.

Files:
ditric.dbf
ditric.shp
ditric.shx
DI_export_lka.xml
DI_export_lka.zip
dsdivi.dbf
dsdivi.shp
dsdivi.shx
provi.dbf
provi.shp
provi.shx


In [5]:
xml_path = os.path.join(
    RAW_LANDSLIDE_DIR,
    "DI_export_lka.xml"
)

output_file = os.path.join(
    PROCESSED_DIR,
    "landslides_sri_lanka.csv"
)

tree = ET.parse(xml_path)
root = tree.getroot()

records = []

for event in root.findall(".//evento/.."):

    fields = {}

    for child in event:
        fields[child.tag] = (child.text or "").strip()

    if fields.get("evento", "").upper() == "LANDSLIDE":

        records.append({
            "record_number": len(records) + 1,
            "serial": fields.get("serial", ""),
            "province": fields.get("name0", ""),
            "district": fields.get("name1", ""),
            "division": fields.get("name2", ""),
            "location": fields.get("lugar", ""),
            "year": fields.get("fechano", ""),
            "month": fields.get("fechames", ""),
            "day": fields.get("fechadia", ""),
            "deaths": fields.get("muertos", ""),
            "injured": fields.get("heridos", ""),
            "missing": fields.get("desaparece", ""),
            "affected_people": fields.get("afectados", ""),
            "houses_destroyed": fields.get("vivdest", ""),
            "houses_affected": fields.get("vivafec", ""),
            "cause": fields.get("causa", ""),
            "agriculture_damage": fields.get("agropecuario", ""),
            "industry_damage": fields.get("industrias", ""),
            "transport_damage": fields.get("transporte", ""),
            "hospital_damage": fields.get("nhospitales", ""),
            "school_damage": fields.get("nescuelas", ""),
            "road_damage_km": fields.get("kmvias", ""),
            "duration": fields.get("duracion", ""),
            "latitude": fields.get("latitude", ""),
            "longitude": fields.get("longitude", ""),
            "source": fields.get("fuentes", ""),
            "uuid": fields.get("uu_id", ""),
            "comments": fields.get("di_comments", "")
        })

df = pd.DataFrame(records)

numeric_columns = [
    "year",
    "month",
    "day",
    "deaths",
    "injured",
    "missing",
    "affected_people",
    "houses_destroyed",
    "houses_affected",
    "agriculture_damage",
    "industry_damage",
    "transport_damage",
    "hospital_damage",
    "school_damage",
    "road_damage_km",
    "duration",
    "latitude",
    "longitude"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["date"] = pd.to_datetime(
    {
        "year": df["year"],
        "month": df["month"],
        "day": df["day"]
    },
    errors="coerce"
)

df = df.sort_values(
    ["date", "district", "division"]
).reset_index(drop=True)

df.to_csv(output_file, index=False)

print("======================================")
print("LANDSLIDE DATASET CREATED")
print("======================================")

print("Total records:", len(df))
print("Unique UUIDs:", df["uuid"].nunique())
print("Date range:", df["date"].min(), "→", df["date"].max())

print("\nDistricts:")
print(df["district"].value_counts())

print("\nSaved:")
print(output_file)

LANDSLIDE DATASET CREATED
Total records: 2256
Unique UUIDs: 2256
Date range: 1974-07-27 00:00:00 → 2019-12-31 00:00:00

Districts:
district
Kandy           769
Badulla         408
Kegalle         287
Nuwara Eliya    264
Ratnapura       255
Kalutara         71
Matale           41
Matara           39
Galle            38
Kurunegala       29
Hambantota       23
Moneragala       13
Colombo          11
Gampaha           6
Polonnaruwa       2
Name: count, dtype: int64

Saved:
c:\Users\User\Documents\GitHub\LankaLandslide-AI\data\processed\landslides_sri_lanka.csv


In [6]:
print(df.head())

   record_number  serial province      district          division  \
0           1992  LA1974  Central  Nuwara Eliya  Ambagamuwakorale   
1           2109  LA1974  Central  Nuwara Eliya  Ambagamuwakorale   
2           2110  LA1974  Central  Nuwara Eliya  Ambagamuwakorale   
3           1994  LA1974  Central  Nuwara Eliya          Kothmale   
4           1993  LA1974  Central  Nuwara Eliya      Nuwara Eliya   

                location  year  month  day  deaths  ...  hospital_damage  \
0             Dicoya(GN)  1974      7   27       9  ...                0   
1          Maskeliya(GN)  1974      7   27       0  ...                0   
2             Hatton(GN)  1974      7   27       0  ...                0   
3         Pundaluoya(GN)  1974      7   27       7  ...                0   
4  Dimbula(GN),Devon(GN)  1974      7   27      26  ...                0   

   school_damage  road_damage_km  duration  latitude longitude         source  \
0              0               0         0     

In [7]:
print(df.shape)

(2256, 29)


In [8]:
print(df.columns.tolist())

['record_number', 'serial', 'province', 'district', 'division', 'location', 'year', 'month', 'day', 'deaths', 'injured', 'missing', 'affected_people', 'houses_destroyed', 'houses_affected', 'cause', 'agriculture_damage', 'industry_damage', 'transport_damage', 'hospital_damage', 'school_damage', 'road_damage_km', 'duration', 'latitude', 'longitude', 'source', 'uuid', 'comments', 'date']


In [9]:
print(df.isna().sum())

record_number         0
serial                0
province              0
district              0
division              0
location              0
year                  0
month                 0
day                   0
deaths                0
injured               0
missing               0
affected_people       0
houses_destroyed      0
houses_affected       0
cause                 0
agriculture_damage    0
industry_damage       0
transport_damage      0
hospital_damage       0
school_damage         0
road_damage_km        0
duration              0
latitude              0
longitude             0
source                0
uuid                  0
comments              0
date                  0
dtype: int64


In [10]:
print(df["district"].value_counts())

district
Kandy           769
Badulla         408
Kegalle         287
Nuwara Eliya    264
Ratnapura       255
Kalutara         71
Matale           41
Matara           39
Galle            38
Kurunegala       29
Hambantota       23
Moneragala       13
Colombo          11
Gampaha           6
Polonnaruwa       2
Name: count, dtype: int64
